In [2]:
import gym
import asset_price_process
import option_price_process
import HedgeEnv
# import machin
import numpy as np
# import torch
import matplotlib.pyplot as plt
# import tensorflow as tf
from asset_price_process import GBM
from option_price_process import BSM
from HedgeEnv import env_hedging
# from machin.frame.algorithms import DQN
# from machin.frame.transition import Transition
# from torch import nn
import time


In [3]:
seed = 345
np.random.seed(seed)
torch.manual_seed(seed)
tf.random.set_seed(seed)

tim = time.time()

mu = 0
dt = 1/5
T = 10
num_steps = T/dt
s_0 = 1
strike_price = s_0
sigma = 0.01
r = 0

apm = GBM(mu=mu, dt=dt, s_0=s_0, sigma=sigma)
opm = BSM(strike_price=strike_price, risk_free_interest_rate=r, volatility=sigma, T=T, dt=dt)
env = env_hedging(asset_price_model=apm, dt=dt, T=T, num_steps=num_steps, trading_cost_para=1,
                     L=1, strike_price=strike_price, int_holdings=False, initial_holding=0, mode="PL",
                  option_price_model=opm)

num_actions = 21


NameError: name 'torch' is not defined

# Build the Deep Hedging Agent

In [ ]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()


In [ ]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [ ]:
class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)


In [ ]:
env.action_space

In [ ]:
# BATCH_SIZE is the number of transitions sampled from the replay buffer
# GAMMA is the discount factor as mentioned in the previous section
# EPS_START is the starting value of epsilon
# EPS_END is the final value of epsilon
# EPS_DECAY controls the rate of exponential decay of epsilon, higher means a slower decay
# TAU is the update rate of the target network
# LR is the learning rate of the ``AdamW`` optimizer
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TAU = 0.005
LR = 1e-4

# Get number of actions from gym action space
n_actions = env.action_space
# Get the number of state observations
state, info = env.reset()
n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)


steps_done = 0


def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)


episode_durations = []


def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Flatten
# from tensorflow.keras.optimizers import Adam
# from rl.agents import DQNAgent
# from rl.policy import BoltzmannQPolicy
# from rl.memory import SequentialMemory

In [ ]:
# def build_agent(model, actions):
#     policy = BoltzmannQPolicy()
#     memory = SequentialMemory(limit=50000, window_length=1)
#     dqn = DQNAgent(model=model, memory=memory, policy=policy, 
#                   nb_actions=actions, nb_steps_warmup=10, target_model_update=1e-2)
#     return dqn



In [ ]:
# dqn = build_agent(model, actions)
# dqn.compile(Adam(lr=1e-3), metrics=['mae'])
# dqn.fit(env, nb_steps=50000, visualize=False, verbose=1)

In [ ]:
# class QNet(nn.Module):
#     def __init__(self, state_dim, hidden_dim, action_num):
#         super(QNet, self).__init__()

#         self.fc1 = nn.Linear(state_dim, hidden_dim)
#         self.fc2 = nn.Linear(hidden_dim, hidden_dim)
#         self.fc3 = nn.Linear(hidden_dim, action_num)

#     def forward(self, state):
#         a = torch.relu(self.fc1(state))
#         a = torch.relu(self.fc2(a))
#         return self.fc3(a)

# qnet = QNet(4, 20, num_actions)
# qnet_t = QNet(4, 20, num_actions)

# dqn = DQN(qnet, qnet_t,
#           torch.optim.Adam,
#           nn.MSELoss(reduction='sum'), discount=0.8, epsilon_decay=0.999, learning_rate=0.001,
#           lr_scheduler=torch.optim.lr_scheduler.StepLR, lr_scheduler_kwargs=[{"step_size": 1000*128}])

# num_eps = 5000
# norm_factor = 10000000


# def test_delta(n=10):
#     rew = []
#     for i in range(n):
#         state = env.reset()
#         done = False
#         state = state[[0, 1, 2, 4]]
#         while not done:
#             action = state[3] - env.h
#             new_state, reward, done = env.step(action)
#             reward = np.sum(reward)
#             new_state = new_state[[0, 1, 2, 4]]
#             reward = -(reward) ** 2 + 1 / 1000 * reward
#             #reward = -(action + env.h - state[3]) ** 2  # remove that
#             rew.append(reward)
#             state = new_state
#     return np.mean(rew)


# def test(n=10):
#     rew = []
#     for i in range(n):
#         state = env.reset()
#         done = False
#         state = state[[0, 1, 2, 4]]
#         while not done:
#             out = dqn.act_discrete({"state": torch.tensor(state, dtype=torch.float32).unsqueeze(0)})
#             action = out.squeeze().detach().numpy() / num_actions
#             new_state, reward, done = env.step(action - env.h)
#             reward = np.sum(reward)
#             new_state = new_state[[0, 1, 2, 4]]
#             reward = -(reward) ** 2 + 1 / 1000 * reward
#             #reward = -(action + env.h - state[3]) ** 2  # remove that
#             if i == 1:
#                 print(action, state[3])
#             rew.append(reward)
#             state = new_state
#     return np.mean(rew)

# rew = []

# for j in range(num_eps):
#     print("episode: ", j)
#     state = env.reset()
#     done = False
#     state = state[[0,1,2,4]]
#     while not done:
#         out = dqn.act_discrete_with_noise({"state": torch.tensor(state, dtype=torch.float32).unsqueeze(0)})
#         action = out.squeeze().detach().numpy()/num_actions - env.h
#         new_state, reward, done = env.step(action)
#         #print(action)
#         #print(reward)
#         reward = np.sum(reward)
#         #print(reward)
#         new_state = new_state[[0, 1, 2, 4]]
#         #print(state)
#         reward = -norm_factor*((reward) ** 2 + 1 / 1000 * reward)
#         rew.append(reward)

#         dqn.store_transition({
#             "state": {"state": torch.tensor(state, dtype=torch.float32).unsqueeze(0)},
#             "action": {"action": out},
#             "next_state": {"state": torch.tensor(new_state, dtype=torch.float32).unsqueeze(0)},
#             "reward": float(reward),  # norm factor
#             "terminal": done
#         })
#         state = new_state

#     if j % 50 == 0 and j != 0:
#         print(test(10), test_delta(10))
#         print("reward: ", np.mean(rew), np.mean(rew)/norm_factor)
#         rew = []

#     if j > 100:
#         for _ in range(int(num_steps)):
#             dqn.update()

# #dqn.save("dqn_model_1000")

# rew_m_l = []
# cost_m_l = []
# for j in range(100):
#     rew = []
#     cost_l = []
#     state = env.reset()
#     done = False
#     state = state[[0,1,2,4]]
#     while not done:
#         out = dqn.act_discrete({"state": torch.tensor(state, dtype=torch.float32).unsqueeze(0)})
#         action = out.squeeze().detach().numpy()/num_actions - env.h
#         new_state, reward, done = env.step(action)
#         cost = reward[1]
#         reward = np.sum(reward)

#         new_state = new_state[[0, 1, 2, 4]]
#         rew.append(reward)
#         cost_l.append(cost)
#         state = new_state
#     rew_m_l.append(np.sum(rew))
#     cost_m_l.append(np.sum(cost_l))
# print("__________")
# print(rew_m_l)
# print(np.mean(rew_m_l), np.std(rew_m_l))
# print("__________")
# print(cost_m_l)
# print(np.mean(cost_m_l), np.std(cost_m_l))

# rew_m_l = []
# cost_m_l = []
# for j in range(100):
#     rew = []
#     cost_l = []
#     state = env.reset()
#     done = False
#     state = state[[0, 1, 2, 4]]
#     while not done:
#         action = state[3] - env.h
#         new_state, reward, done = env.step(action)
#         cost = reward[1]
#         reward = np.sum(reward)

#         new_state = new_state[[0, 1, 2, 4]]
#         rew.append(reward)
#         cost_l.append(cost)
#         state = new_state
#     rew_m_l.append(np.sum(rew))
#     cost_m_l.append(np.sum(cost_l))
# print("__________")
# print(rew_m_l)
# print(np.mean(rew_m_l), np.std(rew_m_l))
# print("__________")
# print(cost_m_l)
# print(np.mean(cost_m_l), np.std(cost_m_l))
# print("__________")
# print(tim - time.time())